# 가공식품의 수와 양성률

- 데이터: `리스테리아ML/data/listeria_df_joined.parquet` (676,537행, 품목 계층 lv1~6 조인본)
- 학습용 품목코드 = `train_prdlst_code`, 가공식품 lv1 = `C0000000000000`
- 양성률(분모=전체 양성) = 해당 행의 검출 수 ÷ 전체 검출 수

In [1]:
import pandas as pd
pd.set_option("display.max_columns", None)
DATA = r"C:\cheon\cheon_wokespace\listeria2026\리스테리아ML\data"
d = pd.read_parquet(f"{DATA}/listeria_df_joined.parquet",
                    columns=["lims_data_key", "train_prdlst_code", "train_prdlst_nm", "lv1_cd", "lv1_nm", "judge_result", "date_basis"])
d["y"] = (d["judge_result"] == "검출").astype(int)
N_POS = int(d["y"].sum())
print(f"전체 {len(d):,}행 · 전체 양성 {N_POS:,}건")

전체 676,537행 · 전체 양성 1,969건


## 0. 'C가 들어간' 코드 = 'C로 시작하는' 코드인지 확인

In [2]:
code = d["train_prdlst_code"].astype("string").fillna("")
has_c = code.str.contains("C")
starts_c = code.str.startswith("C")
print(f"C가 들어간 행 {has_c.sum():,} · C로 시작하는 행 {starts_c.sum():,} · 중간에만 C 있는 행 {(has_c & ~starts_c).sum():,}")

C가 들어간 행 332,934 · C로 시작하는 행 332,934 · 중간에만 C 있는 행 0


## 1. lv1 이 가공식품인데 학습용 품목코드에 C가 없는 행

In [3]:
m = (d["lv1_cd"] == "C0000000000000") & ~has_c
print(f"{m.sum():,}행 · 검출 {d.loc[m, 'y'].sum()}건 · 전체 양성 중 {d.loc[m, 'y'].sum() / N_POS * 100:.2f}%")
d[m].groupby(["train_prdlst_code", "train_prdlst_nm"], observed=True).agg(행수=("y", "size"), 검출=("y", "sum")).reset_index()

342행 · 검출 0건 · 전체 양성 중 0.00%


,train_prdlst_code,train_prdlst_nm,행수,검출
0,X0100027500000,기타제조가공식품류(자가규격),342,0


## 2. 학습용 품목코드에 C가 들어간 행의 양성률

In [4]:
g = d[has_c]
pos = int(g["y"].sum())
pd.DataFrame([{"행수": len(g), "검출": pos,
               "양성률(분모=전체 양성, %)": round(pos / N_POS * 100, 2),
               "그룹 내 검출률(분모=그룹 행, %)": round(g["y"].mean() * 100, 4)}])

,행수,검출,"양성률(분모=전체 양성, %)","그룹 내 검출률(분모=그룹 행, %)"
0,332934,449,22.8,0.1349


## 3. 학습용 품목코드에 C가 들어간 행의 기준일 출처(`date_basis`) 비율

In [5]:
db = g["date_basis"].astype("string").fillna("(결측)")
t = g.groupby(db)["y"].agg(행수="size", 검출="sum")
t["행 비율(C 행 중, %)"] = (t["행수"] / len(g) * 100).round(2)
t["양성률(분모=전체 양성, %)"] = (t["검출"] / N_POS * 100).round(2)
t["그룹 내 검출률(%)"] = (t["검출"] / t["행수"] * 100).round(4)
t.sort_values("행수", ascending=False)

,행수,검출,"행 비율(C 행 중, %)","양성률(분모=전체 양성, %)",그룹 내 검출률(%)
date_basis,,,,,
제조일,222596,288,66.86,14.63,0.1294
의뢰일,71675,98,21.53,4.98,0.1367
선적일,38663,63,11.61,3.20,0.1629
